# Fig6, stage 00: read the wells, label them, concatenate

Experiment LPHO015, the twenty experiments behind Figure 6.  The pipeline is the same as for the numbered loops; only the input experiment and the downstream figure panels differ.

## Inputs

| file | produced by | holds |
|---|---|---|
| `metadata_results.csv` | this folder | culture conditions, one row per experiment |
| `$LABCOMPASS_FCS_ROOT/LPHO015` | acquisition, outside the repository | one unmixed `.fcs` file per well |
| `wells.txt` | this folder | the wells to read, in the order that fixes the well to experiment mapping |
| `live.npz` | this folder | gates for the live-cell count per well |
| `counts.npz` | this folder | gates for the counting-bead count per well |

## Outputs

| file | holds |
|---|---|
| `Fig6.h5ad` | every cell of every well, labelled, with the antibody channels only |
| `Fig6_500k.h5ad` | about 500,000 of those cells, spread evenly over the experiments |

Run the cells from top to bottom.  Nothing below reads a file that a later cell
writes, so `Restart Kernel and Run All Cells` reproduces both outputs.

## Setup

Every path and every number that this notebook uses is set here.  The only
import from `labcompass.py` is the gate reader, which is long enough to be worth
sharing; everything else this notebook does is written out where it happens.

In [ ]:
import gc
import glob
import os

import anndata as ann
import numpy as np
import pandas as pd
import pytometry as pm

import sys

sys.path.append("..")  # labcompass.py sits beside the loop folders
from labcompass import apply_saved_gates, downsample

# ---------------------------------------------------------------- configuration
# Where the unmixed .fcs files live.  They are far too large to commit, so the
# repository only records the path.  Override it without editing this notebook
# by setting the environment variable LABCOMPASS_FCS_ROOT.
FCS_ROOT = os.environ.get("LABCOMPASS_FCS_ROOT", '/rds/project/rds-SDzz0CATGms/unsorted/for_Juan_Licyel')
DATA_PATH = os.path.join(FCS_ROOT, 'LPHO015')
METADATA_CSV = 'metadata_results.csv'

# One experiment per (plate, plate row) pair, in the order in which the
# wells are read.  The order of this list is therefore part of the data:
# reordering it relabels every cell.
EXPERIMENTS = [
    '262', '263', '264', '265', '266', '267',
    '270', '271', '273', '276', '275', '261',
    '279', '277', '280', '269', '268', '272',
    '278', '274',
]

# Gates used per well to count live cells and counting beads.  Both are
# applied to the raw well, before the scatter channels move into .obs.
LIVE_GATES = "live.npz"
COUNT_GATES = "counts.npz"

WELL_LIST = "wells.txt"

OUT_ALL = "Fig6.h5ad"
OUT_SUBSET = "Fig6_500k.h5ad"

# Cap the downsampled file at this many cells in total, spread evenly over the
# experiments.  It exists so the gating notebook has something small to plot.
DESIRED_CELL_NUMBER = 500000
DOWNSAMPLE_SEED = 0

# The live/dead channel.  It is copied into .obs before it is dropped, because
# the live gate is drawn against it.
LIVE_DEAD_CHANNEL = "Live_Dead : 7-AAD - Area"

# Channels that carry no antibody signal.  The seven "[AF color N]" channels are
# the unmixed autofluorescence components the spectral cytometer reports.  CD184
# is dropped because its staining failed in this panel.
CHANNELS_TO_DROP = [
    "CD184 : PerCP-eFluor710 - Area",
    LIVE_DEAD_CHANNEL,
    "[AF color 1] - Area",
    "[AF color 2] - Area",
    "[AF color 3] - Area",
    "[AF color 4] - Area",
    "[AF color 5] - Area",
    "[AF color 6] - Area",
    "[AF color 7] - Area",
]

## The wells to read

The wells are listed in a file rather than found by pattern, because the order
in which they are read decides which experiment number each well is given.  A
pattern search would order the wells by path, and that order does not match the
plate layout of this experiment.

In [ ]:
# Each line of wells.txt is a path relative to FCS_ROOT.
with open(WELL_LIST) as handle:
    files = [os.path.join(FCS_ROOT, line.strip()) for line in handle if line.strip()]

missing = [f for f in files if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(
        f"{len(missing)} of {len(files)} wells are missing, the first is {missing[0]}"
    )
print(f"{len(files)} wells to read")

## The condition table

One row per experiment, holding the culture conditions.  The index is forced to
text.  Without that, pandas infers an integer index when every experiment number
happens to be numeric, and a later lookup with a string key silently finds
nothing.

In [ ]:
metadata = pd.read_csv(METADATA_CSV, sep="\t").set_index("experiment_number")
metadata.index = metadata.index.astype(str)
condition_columns = metadata.columns.tolist()

print(f"{len(metadata)} experiments, {len(condition_columns)} condition columns")
metadata.head()

## Read each well and label it

The wells are read in the order of the list above.  Each new combination of
plate name and plate row starts a new experiment, taking the next entry of
`EXPERIMENTS`, and the wells inside one such combination are numbered as
replicates 1, 2, 3 and so on.  The order of the well list and the order of
`EXPERIMENTS` therefore have to agree, which is why both are written out
explicitly rather than derived.

Two counts are recorded per well before any channel is dropped.  `cell_counts`
is the number of live cells, from the gates in `live.npz`.  `count_beads` is the
number of counting beads, from the gates in `counts.npz`.  Their ratio converts
an event count into an absolute cell number.

Four pieces of metadata come from the `.fcs` file itself, written there by the
cytometer: the acquisition date, the cytometer make, its serial number and the
well identifier.  The culture conditions come from the condition table.

`pytometry.pp.split_signal` moves the scatter and time channels from `.X` into
`.obs` and keeps only the area measurements in `.X`.

In [ ]:
adatas = []
skipped = []
previous_group = None
experiment_index = -1
replicate = 0

for path in files:
    adata = pm.io.read_fcs(path)
    adata.obs.index = adata.obs.index.astype(str)
    # var_names has to hold the channel names, so a gate saved against a channel
    # can be applied before split_signal moves the scatter channels into .obs.
    adata.var_names = adata.var["marker"].astype(str)
    if adata.n_obs == 0:
        continue

    # A group is one plate row of one plate, and holds the replicates of one
    # experiment.  The wells of a group have to be contiguous in `files`.
    group = adata.uns["meta"]["platename"] + adata.uns["meta"]["wellid"][0]
    if group != previous_group:
        previous_group = group
        experiment_index += 1
        replicate = 1
    else:
        replicate += 1

    if experiment_index >= len(EXPERIMENTS):
        raise IndexError(
            f"well {path} starts group number {experiment_index + 1}, but "
            f"EXPERIMENTS only has {len(EXPERIMENTS)} entries; the well list and "
            "EXPERIMENTS have gone out of step"
        )

    experiment_number = EXPERIMENTS[experiment_index]
    if experiment_number not in metadata.index:
        skipped.append((path, f"experiment {experiment_number} not in {METADATA_CSV}"))
        continue

    # Counted on the raw well, while the scatter channels are still in .X.
    live_cells, _ = apply_saved_gates(adata, LIVE_GATES)
    beads, _ = apply_saved_gates(adata, COUNT_GATES)
    adata.obs["cell_counts"] = live_cells.n_obs
    adata.obs["count_beads"] = beads.n_obs

    adata.obs["experiment_number"] = experiment_number
    adata.obs["replicate"] = replicate
    adata.obs["date"] = adata.uns["meta"]["date"]
    adata.obs["cytometer"] = adata.uns["meta"]["cyt"]
    adata.obs["cytometer_serial_no"] = adata.uns["meta"]["cytsn"]
    adata.obs["well_id"] = adata.uns["meta"]["smno"]
    adata.obs[condition_columns] = metadata.loc[str(experiment_number)].tolist()

    pm.pp.split_signal(adata, var_key="channel", option="area", data_type="facs")
    adatas.append(adata)

del adata
gc.collect()

print(f"{len(adatas)} wells kept, {len(skipped)} skipped, "
      f"{experiment_index + 1} of {len(EXPERIMENTS)} experiments seen")
for path, reason in skipped:
    print("  skipped", path, "|", reason)

## Concatenate

`join="inner"` keeps only the channels present in every well.
`index_unique="-"` appends each well's position to its cell names, so no two
cells share a name and later steps can index by name safely.

In [ ]:
adata_all = ann.concat(
    adatas,
    join="inner",
    merge="same",
    uns_merge="unique",
    label="batch",
    index_unique="-",
)
del adatas
gc.collect()
adata_all

## Move the live/dead channel, drop the channels we do not analyse, rename the rest

A channel name from the unmixing software has the form
`"<antibody> : <fluorochrome> - <parameter>"`, for example
`"CD34 : APC - Area"`.  It is split so `.var_names` holds the plain antibody
name and `.var["antibody"]` keeps it as a category.

In [ ]:
# The live gate is drawn against 7-AAD, and gating reads from .obs, so the
# channel has to be copied across before it is dropped from .X below.
values = adata_all[:, LIVE_DEAD_CHANNEL].X
adata_all.obs["7-AAD"] = np.asarray(
    values.toarray() if hasattr(values, "toarray") else values
).ravel()

keep = ~np.isin(adata_all.var["marker"].values, CHANNELS_TO_DROP)
adata_all = adata_all[:, keep]

markers = list(adata_all.var["marker"].values)
antibodies = [marker.split(" :")[0] for marker in markers]
fluorochromes = [marker.split(": ")[1].split(" -")[0] for marker in markers]

duplicates = sorted({a for a in antibodies if antibodies.count(a) > 1})
if duplicates:
    raise ValueError(f"channels reduce to duplicate antibody names: {duplicates}")

adata_all.var = adata_all.var.copy()
adata_all.var["antibody"] = pd.Categorical(antibodies)
adata_all.var_names = antibodies

print("antibodies:   ", antibodies)
print("fluorochromes:", fluorochromes)
adata_all

## Write the full object

This is the input of stage 02.

In [ ]:
adata_all.write(OUT_ALL)
print(f"wrote {OUT_ALL}: {adata_all.n_obs:,} cells x {adata_all.n_vars} antibodies")

## Write the downsampled object

Each experiment is capped at the same number of cells.  An experiment with fewer
cells than the cap is kept whole, so the total comes out below the target.

In [ ]:
target_size = int(DESIRED_CELL_NUMBER / adata_all.obs["experiment_number"].nunique())
print(f"cap of {target_size:,} cells per experiment")

adata_small = downsample(
    adata_all, obs="experiment_number", target_size=target_size, seed=DOWNSAMPLE_SEED
)
adata_small.write(OUT_SUBSET)
print(f"wrote {OUT_SUBSET}: {adata_small.n_obs:,} cells")